In [0]:
%run "../../commons/commons_imports"

In [0]:
df_item_bronze = read(
    base_path=BRONZE_PATH,
    table_name=TS_ITEM,
    recursive_by_year=True
)

In [0]:
df_item_silver = (
    df_item_bronze

    # =====================================================
    # Conversão de tipos
    # =====================================================

    .withColumn("NU_ANO_AVALIACAO", col("NU_ANO_AVALIACAO").cast("int"))
    .withColumn("CO_UF", col("CO_UF").cast("int"))
    .withColumn("CO_BLOCO", col("CO_BLOCO").cast("int"))
    .withColumn("NU_POSICAO", col("NU_POSICAO").cast("int"))
    .withColumn("CO_ITEM", col("CO_ITEM").cast("int"))

    .withColumn("TP_SERIE", col("TP_SERIE").cast("int"))
    .withColumn("TP_DISCIPLINA", col("TP_DISCIPLINA").cast("int"))
    .withColumn("TP_RESPOSTA_ITEM", col("TP_RESPOSTA_ITEM").cast("int"))
    .withColumn("TP_MODELO_TRI", col("TP_MODELO_TRI").cast("int"))
    .withColumn("IN_ITEM_COMUM", col("IN_ITEM_COMUM").cast("int"))

    .withColumn("NU_PARAM_A", col("NU_PARAM_A").cast("double"))
    .withColumn("NU_PARAM_B", col("NU_PARAM_B").cast("double"))
    .withColumn("NU_PARAM_C", col("NU_PARAM_C").cast("double"))

    .withColumn("NU_PARAM_B1", col("NU_PARAM_B1").cast("double"))
    .withColumn("NU_PARAM_B2", col("NU_PARAM_B2").cast("double"))
    .withColumn("NU_PARAM_B3", col("NU_PARAM_B3").cast("double"))
    .withColumn("NU_PARAM_B4", col("NU_PARAM_B4").cast("double"))

    # =====================================================
    # Padronização
    # =====================================================

    .withColumn("SG_UF", upper(trim(col("SG_UF"))))

    .withColumn(
        "NU_DESCRITOR_HABILIDADE",
        upper(trim(col("NU_DESCRITOR_HABILIDADE")))
    )

    .withColumn(
        "DS_GABARITO",
        upper(trim(col("DS_GABARITO")))
    )

    # =====================================================
    # Colunas descritivas
    # =====================================================

    .withColumn(
        "DS_SERIE",
        when(col("TP_SERIE") == 2, "2º Ano do Ensino Fundamental")
    )

    .withColumn(
        "DS_DISCIPLINA",
        when(col("TP_DISCIPLINA") == 1, "Língua Portuguesa")
        .when(col("TP_DISCIPLINA") == 2, "Matemática")
    )

    .withColumn(
        "DS_RESPOSTA_ITEM",
        when(col("TP_RESPOSTA_ITEM") == 1, "Produção Textual")
        .when(col("TP_RESPOSTA_ITEM") == 2, "Resposta Construída")
        .when(col("TP_RESPOSTA_ITEM") == 3, "Resposta Objetiva")
        .when(col("TP_RESPOSTA_ITEM") == 4, "Resposta Construída com Quesito")
    )

    .withColumn(
        "DS_MODELO_TRI",
        when(col("TP_MODELO_TRI") == 1, "M2PL")
        .when(col("TP_MODELO_TRI") == 2, "M3PL")
        .when(col("TP_MODELO_TRI") == 3, "MRG")
        .when(col("TP_MODELO_TRI") == 4, "M3P")
    )

    .withColumn(
        "DS_ITEM_COMUM",
        when(col("IN_ITEM_COMUM") == 1, "Sim")
        .otherwise("Não")
    )

    # =====================================================
    # Chave técnica 
    # =====================================================

    .withColumn(
        "SK_ITEM",
        sha2(
            concat_ws("|", col("CO_ITEM")),
            256
        )
    )

    # =====================================================
    # Colunas técnicas
    # =====================================================

    .withColumn("DT_PROCESSAMENTO", current_date())
    .withColumn("TS_PROCESSAMENTO", current_timestamp())

    # =====================================================
    # Remove duplicados
    # =====================================================

    .dropDuplicates([
        "CO_ITEM"
    ])
)

In [0]:
df_item_silver_selected = df_item_silver.select(

    # =====================================================
    # Chave técnica
    # =====================================================

    "SK_ITEM",

    # =====================================================
    # Chaves de negócio
    # =====================================================

    "CO_ITEM",

    "NU_ANO_AVALIACAO",
    "ANO_REFERENCIA",

    "CO_UF",
    "SG_UF",

    "CO_BLOCO",
    "NU_POSICAO",

    "TP_SERIE",
    "DS_SERIE",

    "TP_DISCIPLINA",
    "DS_DISCIPLINA",

    # =====================================================
    # Informações do Item
    # =====================================================

    "NU_DESCRITOR_HABILIDADE",
    "DS_GABARITO",

    "TP_RESPOSTA_ITEM",
    "DS_RESPOSTA_ITEM",

    "TP_MODELO_TRI",
    "DS_MODELO_TRI",

    "IN_ITEM_COMUM",
    "DS_ITEM_COMUM",

    # =====================================================
    # Parâmetros TRI
    # =====================================================

    "NU_PARAM_A",
    "NU_PARAM_B",
    "NU_PARAM_C",

    "NU_PARAM_B1",
    "NU_PARAM_B2",
    "NU_PARAM_B3",
    "NU_PARAM_B4",

    # =====================================================
    # Auditoria
    # =====================================================

    "DT_PROCESSAMENTO",
    "TS_PROCESSAMENTO"
)

In [0]:
write_delta(
    df=df_item_silver_selected,
    base_path=SILVER_PATH,
    table_name=TS_ITEM,
    merge_keys=["SK_ITEM"]
)